# Phase 3 

### Chargement du dataset

In [4]:
import time
import numpy as np
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import cross_val_score, KFold, StratifiedKFold, LeaveOneOut
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
data = load_breast_cancer()

X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")

print("Classes :", data.target_names)
print("\nRépartition de la cible :")
print(y.value_counts(normalize=True).sort_index().round(3))

Classes : ['malignant' 'benign']

Répartition de la cible :
target
0    0.373
1    0.627
Name: proportion, dtype: float64


### validation croisée k-fold

#### cas normal 

In [5]:
modele = make_pipeline(StandardScaler(),LogisticRegression(max_iter=5000, random_state=42))
def evaluer_en_cross_val(modele, X, y, k=5):
    

    scores = cross_val_score(
        modele,
        X,
        y,
        cv=k,
        scoring="accuracy"
    )

    moyenne = np.mean(scores)
    ecart_type = np.std(scores, ddof=1)

    print("Scores par fold :", np.round(scores, 3))
    print(f"Moyenne : {moyenne:.3f} | Écart-type : {ecart_type:.3f}", end="")

    if ecart_type < 0.02:
        print(" -> modèle stable")
    else:
        print(" -> modèle moins stable")

    return scores
scores_cv_5 = evaluer_en_cross_val(
    modele=modele,
    X=X,
    y=y,
    k=5
)

Scores par fold : [0.982 0.982 0.974 0.974 0.991]
Moyenne : 0.981 | Écart-type : 0.007 -> modèle stable


#### Cas limite leave-one-out

In [8]:
loo = LeaveOneOut()

debut = time.time()

scores_leave_one_out = cross_val_score(
    modele,
    X,
    y,
    cv=loo,
    scoring="accuracy"
)

fin = time.time()
temps_execution = fin - debut

print(f"Moyenne : {np.mean(scores_leave_one_out):.3f}")
print(f"Écart-type : {np.std(scores_leave_one_out, ddof=1):.3f}")
print(f"Temps d'exécution : {temps_execution:.2f} secondes")

Moyenne : 0.979
Écart-type : 0.144
Temps d'exécution : 4.29 secondes


#### Cas adversarial 95/5 

#### Créer un dataset déséquilibré 95/5

In [9]:
rng = np.random.default_rng(42)

X_adversarial = pd.DataFrame(
    rng.normal(size=(100, 5)),
    columns=["feature_1", "feature_2", "feature_3", "feature_4", "feature_5"]
)

y_adversarial = pd.Series(
    [0] * 95 + [1] * 5,
    name="target"
)

print("Taille du dataset adversarial :", len(X_adversarial))

print("\nRépartition initiale des classes :")
print(y_adversarial.value_counts(normalize=True).sort_index().round(3))

Taille du dataset adversarial : 100

Répartition initiale des classes :
target
0    0.95
1    0.05
Name: proportion, dtype: float64


#### Répartition avec KFold

In [12]:
modele_adversarial = DecisionTreeClassifier(random_state=42)
kfold_non_stratifie = KFold(
    n_splits=5,
    shuffle=False
)

kfold_stratifie = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)
print("Répartition des classes avec KFold sans stratification :\n")

for numero_fold, (_, indices_test) in enumerate(kfold_non_stratifie.split(X_adversarial), start=1):
    repartition_fold = y_adversarial.iloc[indices_test].value_counts().sort_index()
    print(f"Fold {numero_fold} :")
    print(repartition_fold)
    print()

Répartition des classes avec KFold sans stratification :

Fold 1 :
target
0    20
Name: count, dtype: int64

Fold 2 :
target
0    20
Name: count, dtype: int64

Fold 3 :
target
0    20
Name: count, dtype: int64

Fold 4 :
target
0    20
Name: count, dtype: int64

Fold 5 :
target
0    15
1     5
Name: count, dtype: int64



#### Répartition avec StratifiedKFold

In [13]:
print("Répartition des classes avec StratifiedKFold :\n")

for numero_fold, (_, indices_test) in enumerate(
    kfold_stratifie.split(X_adversarial, y_adversarial),
    start=1
):
    repartition_fold = y_adversarial.iloc[indices_test].value_counts().sort_index()
    print(f"Fold {numero_fold} :")
    print(repartition_fold)
    print()

Répartition des classes avec StratifiedKFold :

Fold 1 :
target
0    19
1     1
Name: count, dtype: int64

Fold 2 :
target
0    19
1     1
Name: count, dtype: int64

Fold 3 :
target
0    19
1     1
Name: count, dtype: int64

Fold 4 :
target
0    19
1     1
Name: count, dtype: int64

Fold 5 :
target
0    19
1     1
Name: count, dtype: int64



#### Comparer les scores

#### Scores avec KFold

In [14]:
scores_kfold = cross_val_score(
    modele_adversarial,
    X_adversarial,
    y_adversarial,
    cv=kfold_non_stratifie,
    scoring="accuracy"
)

print("Scores avec KFold sans stratification :", np.round(scores_kfold, 3))
print(f"Moyenne : {np.mean(scores_kfold):.3f}")
print(f"Écart-type : {np.std(scores_kfold, ddof=1):.3f}")

Scores avec KFold sans stratification : [0.95 0.9  0.9  0.9  0.75]
Moyenne : 0.880
Écart-type : 0.076


#### Scores avec StratifiedKFold

In [15]:
scores_stratified = cross_val_score(
    modele_adversarial,
    X_adversarial,
    y_adversarial,
    cv=kfold_stratifie,
    scoring="accuracy"
)

print("Scores avec StratifiedKFold :", np.round(scores_stratified, 3))
print(f"Moyenne : {np.mean(scores_stratified):.3f}")
print(f"Écart-type : {np.std(scores_stratified, ddof=1):.3f}")

Scores avec StratifiedKFold : [0.9  0.8  0.8  0.95 0.9 ]
Moyenne : 0.870
Écart-type : 0.067
